<a href="https://colab.research.google.com/github/Rakshit-Gupta-77/Machine-Learning-Journey/blob/main/titanic_without_using_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Titanic — Preprocessing Without Using Pipeline

This notebook follows the original Titanic workflow **without `Pipeline` or `ColumnTransformer`**.

We will:
1. Load and clean the Titanic dataset
2. Split into train/test data
3. Impute missing `Age` and `Embarked`
4. One-hot encode `Sex` and `Embarked`
5. Combine the transformed features manually
6. Train a Decision Tree
7. Evaluate accuracy
8. Save the encoders and model

> **Important:** Fit preprocessing steps only on training data. Use `transform()` on test data to avoid data leakage.

In [19]:
import os
import pickle
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [20]:
df = pd.read_csv('train.csv')
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 1. Remove unnecessary columns

The original notebook removes:

- `PassengerId`
- `Name`
- `Ticket`
- `Cabin`

These columns are not used in the preprocessing workflow below.

In [21]:
df.drop(
    columns=['PassengerId', 'Name', 'Ticket', 'Cabin'],
    inplace=True
)

df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


## 2. Train/Test Split

`Survived` is the target (`y`).

Everything else is used as the input features (`X`).

We use `stratify=y` so the class proportion is kept approximately similar in train and test sets.

In [22]:
X = df.drop(columns=['Survived'])
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (712, 7)
X_test : (179, 7)
y_train: (712,)
y_test : (179,)


In [23]:
X_train.head(2)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
692,3,male,NaN,0,0,56.4958,S
481,2,male,NaN,0,0,0.0000,S


In [24]:
y_train.head()

,Survived
692,1
481,0
527,0
855,1
801,1


## 3. Check missing values

We mainly need to handle missing values in:

- `Age`
- `Embarked`

We will learn the imputation values from the **training data only**.

In [25]:
df.isnull().sum()

,0
Survived,0
Pclass,0
Sex,0
Age,177
SibSp,0
Parch,0
Fare,0
Embarked,2


## 4. Imputation

### Age
`SimpleImputer()` uses the **mean** by default.

### Embarked
For a categorical column, we use the most frequent category.

Notice the train/test rule:

```python
fit_transform()  # training data
transform()      # test data
```

We do **not** fit the imputers again on the test set.

In [26]:
si_age = SimpleImputer(strategy='mean')
si_embarked = SimpleImputer(strategy='most_frequent')

X_train_age = si_age.fit_transform(X_train[['Age']])
X_test_age = si_age.transform(X_test[['Age']])

X_train_embarked_imputed = si_embarked.fit_transform(X_train[['Embarked']])
X_test_embarked_imputed = si_embarked.transform(X_test[['Embarked']])

print("Age train shape:", X_train_age.shape)
print("Age test shape :", X_test_age.shape)
print("Embarked train shape:", X_train_embarked_imputed.shape)
print("Embarked test shape :", X_test_embarked_imputed.shape)

Age train shape: (712, 1)
Age test shape : (179, 1)
Embarked train shape: (712, 1)
Embarked test shape : (179, 1)


In [27]:
X_train_embarked_imputed[:5]

array([['S'],
       ['S'],
       ['S'],
       ['S'],
       ['S']], dtype=object)

## 5. One-Hot Encoding

We need to encode:

- `Sex`
- `Embarked`

Both are **nominal categorical features**, so One-Hot Encoding is appropriate.

We use `handle_unknown='ignore'` so an unseen category in the test/new data does not cause an error.

### Current scikit-learn API

Use:

```python
sparse_output=False
```

instead of the older:

```python
sparse=False
```

In [28]:
ohe_sex = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore'
)

ohe_embarked = OneHotEncoder(
    sparse_output=False,
    handle_unknown='ignore'
)

X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_test_sex = ohe_sex.transform(X_test[['Sex']])

X_train_embarked = ohe_embarked.fit_transform(X_train_embarked_imputed)
X_test_embarked = ohe_embarked.transform(X_test_embarked_imputed)

print("Sex encoded shape:", X_train_sex.shape)
print("Embarked encoded shape:", X_train_embarked.shape)

print("Sex columns:", ohe_sex.get_feature_names_out(['Sex']))
print("Embarked columns:", ohe_embarked.get_feature_names_out(['Embarked']))

Sex encoded shape: (712, 2)
Embarked encoded shape: (712, 3)
Sex columns: ['Sex_female' 'Sex_male']
Embarked columns: ['Embarked_C' 'Embarked_Q' 'Embarked_S']


In [29]:
X_train_embarked[:5]

array([[0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.],
       [0., 0., 1.]])

## 6. Keep the remaining columns

We have already transformed:

- `Sex`
- `Age`
- `Embarked`

So remove those three columns from the original DataFrames.

The remaining numerical columns are kept as they are.

In [30]:
X_train_rem = X_train.drop(columns=['Sex', 'Age', 'Embarked'])
X_test_rem = X_test.drop(columns=['Sex', 'Age', 'Embarked'])

print("Remaining train columns:")
print(X_train_rem.columns.tolist())

Remaining train columns:
['Pclass', 'SibSp', 'Parch', 'Fare']


## 7. Manually combine all transformed features

This is the key idea of **preprocessing without Pipeline**.

We manually concatenate:

```text
Remaining numerical features
        +
Imputed Age
        +
Encoded Sex
        +
Encoded Embarked
```

`np.concatenate(..., axis=1)` joins them column-wise.

In [31]:
X_train_transformed = np.concatenate(
    (
        X_train_rem.to_numpy(),
        X_train_age,
        X_train_sex,
        X_train_embarked
    ),
    axis=1
)

X_test_transformed = np.concatenate(
    (
        X_test_rem.to_numpy(),
        X_test_age,
        X_test_sex,
        X_test_embarked
    ),
    axis=1
)

print("X_train transformed shape:", X_train_transformed.shape)
print("X_test transformed shape :", X_test_transformed.shape)

X_train transformed shape: (712, 10)
X_test transformed shape : (179, 10)


In [32]:
X_test_transformed.shape

(179, 10)

## 8. Train a Decision Tree

Now all input features are numerical, so we can train the classifier.

In [33]:
clf = DecisionTreeClassifier(random_state=42)

clf.fit(X_train_transformed, y_train)

DecisionTreeClassifier(random_state=42)

## 9. Make predictions

In [34]:
y_pred = clf.predict(X_test_transformed)

y_pred[:10]

array([0, 0, 0, 0, 1, 0, 1, 1, 0, 0])

## 10. Evaluate the model

We use **accuracy** to compare predictions with the true test labels.

In [35]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8100558659217877


## 11. Save the preprocessing objects and model

These objects are needed later when making predictions on new Titanic data.

We create the `models` folder first so `pickle.dump()` does not fail if the folder does not exist.

In [36]:
os.makedirs('models', exist_ok=True)

with open('models/si_age.pkl', 'wb') as f:
    pickle.dump(si_age, f)

with open('models/si_embarked.pkl', 'wb') as f:
    pickle.dump(si_embarked, f)

with open('models/ohe_sex.pkl', 'wb') as f:
    pickle.dump(ohe_sex, f)

with open('models/ohe_embarked.pkl', 'wb') as f:
    pickle.dump(ohe_embarked, f)

with open('models/clf.pkl', 'wb') as f:
    pickle.dump(clf, f)

print("Saved preprocessing objects and model in the 'models' folder.")

Saved preprocessing objects and model in the 'models' folder.


## Quick Revision

### Preprocessing order

```text
Raw Titanic Data
      ↓
Train/Test Split
      ↓
Missing Values
      ↓
SimpleImputer
      ↓
Categorical Encoding
      ↓
OneHotEncoder
      ↓
Manual Concatenation
      ↓
Decision Tree
      ↓
Prediction
      ↓
Accuracy
```

### Important rule

```text
TRAIN → fit_transform()
TEST  → transform()
```

Never fit the preprocessing objects separately on the test set.

### What each tool does

| Tool | Purpose |
|---|---|
| `SimpleImputer` | Handles missing values |
| `OneHotEncoder` | Converts categorical values into binary columns |
| `np.concatenate` | Manually combines transformed features |
| `DecisionTreeClassifier` | Trains the ML model |
| `accuracy_score` | Measures prediction accuracy |
| `pickle` | Saves preprocessing objects and model |